# Week 7 - Outlier Detection and Data Quality
Calculate IQR boundaries and flag outliers (specifically add flags for ClosePrice, LivingArea, and DaysOnMarket)

In [ ]:
import pandas as pd
from pathlib import Path

In [2]:
# load in dataset
data_dir = Path('../data/feature_engineered')

sold = pd.read_csv(
    data_dir / 'sold_feature_engineered.csv',
    low_memory=False
)

print('Sold Shape -', sold.shape) # this should output as (413853, 104)
sold.head()

Sold Shape - (413853, 104)


,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,close_to_original_list_ratio,Year,Month,YrMo,listing_to_contract_days,contract_to_close_days,elementary_school_district,high_school_district,unified_school_district,school_district_matched_flag
0,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,2024-01-26,240000.0,...,0.480962,2024,1,2024-01,777.0,65.0,San Mateo-Foster City,San Mateo Union High,NaN,True
1,NaN,False,NaN,NaN,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,815000.0,...,1.072510,2024,1,2024-01,114.0,919.0,National Elementary,Sweetwater Union High,NaN,True
2,NaN,False,NaN,NaN,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,810000.0,...,1.094743,2024,1,2024-01,255.0,778.0,National Elementary,Sweetwater Union High,NaN,True
3,NaN,True,NaN,NaN,False,2100000.0,1067652762,jenniferarielkennedy@gmail.com,2024-01-02,2100000.0,...,1.000000,2024,1,2024-01,0.0,48.0,NaN,NaN,San Luis Coastal Unified,True
4,"Tile,Wood",True,NaN,NaN,NaN,NaN,1061988701,dannyduong@gmail.com,2024-01-31,2340000.0,...,NaN,2024,1,2024-01,33.0,-33.0,Evergreen Elementary,East Side Union High,NaN,True


## 1) Data check

Before applying IQR and filtering, I wanted to look at the distributions and check whether any of the three columns had missing values potentially from last week.

In [3]:
# cehck for any missing values in the key numeric fields we need to apply IQR filtering to
# DO NOT delete records outright
numeric_columns = [
    'ClosePrice',
    'LivingArea',
    'DaysOnMarket'
]

# convert columns to numeric if needed
for column in numeric_columns:
    sold[column] = pd.to_numeric(sold[column], errors='coerce')

display(sold[numeric_columns].describe())

print('Missing Values:')
print(sold[numeric_columns].isna().sum())

,ClosePrice,LivingArea,DaysOnMarket
count,4.138530e+05,4.136190e+05,413853.000000
mean,1.193067e+06,1.904507e+03,37.357994
std,6.290538e+06,2.648261e+04,53.580019
min,1.150000e+00,1.000000e+00,0.000000
25%,5.750000e+05,1.248000e+03,8.000000
50%,8.230000e+05,1.643000e+03,18.000000
75%,1.300000e+06,2.219000e+03,48.000000
max,9.895000e+08,1.702132e+07,12430.000000


Missing Values:
ClosePrice        0
LivingArea      234
DaysOnMarket      0
dtype: int64


I'll leave these observations in the dataset for now because a missing value isn't necessarily an outlier.

*(Inspired by Ziye's explanation from last meeting)*

The maximum values also seem unusually large compared to the median, so I wanted to look at the percentiles before calculating the IQR boundaries.Here, I examine the percentiles before performing any filtering for IQR or flagging outliers.

In [4]:
# check percentiles to understand extreme values in the dataset
percentile_summary = sold[numeric_columns].quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

percentile_summary

,ClosePrice,LivingArea,DaysOnMarket
0.01,203000.0,609.0,0.0
0.05,340000.0,840.0,1.0
0.25,575000.0,1248.0,8.0
0.50,823000.0,1643.0,18.0
0.75,1300000.0,2219.0,48.0
0.95,2850000.0,3560.0,132.0
0.99,5550000.0,5283.0,231.0


## 2) Use IQR to identify + remove values outside of the boundaries

Implement the Interquartile Range (IQR) method removes records that fall outside a defined statistical range which is given as: $Q3 - Q1$
$$\text{Lower} = Q1 - 1.5(IQR)$$
$$\text{Upper} = Q3 + 1.5(IQR)$$

There's the `quantile()` function from panads where we can call the method on a column to calculate the quantiles.

Start off with creating a function that calculates all the thresholds for IQR using the original sold dataset and then flags the outliers that flal outside of the quantiles.

In [23]:
# calculate IQR boundaries and identify outliers

def flag_iqr(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)

    iqr = q3 - q1
    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)

    # missing values not automatically considered outliers
    flags = df[column].notna() & (
        (df[column] < lower) | (df[column] > upper)
    )

    return flags, q1, q3, iqr, lower, upper


## 3) Flag outliers
Now actually apply the function so we can find the IQR boundaries and flag the outliers for these specific columns: (ClosePrice, LivingArea, DaysOnMarket)

*Use the IQR boundary function that was made earlier*

In [ ]:
iqr_summary = []

for column in numeric_columns:
    flags, q1, q3, iqr, lower, upper = flag_iqr(sold, column)

    # add outlier flags to the original dataset
    # SHOULD ONLY CREATE 3 COLUMNS: [outlier_closeprice, outlier_livingarea, outlier_daysonmarket]
    sold['outlier_' + column.lower()] = flags


    # record the IQR boundaries and outlier counts for report summary
    iqr_summary.append({
        'Column': column,
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'LowerBound': lower,
        'UpperBound': upper,
        'OutlierCount': flags.sum(),
        'OutlierPercent': flags.mean() * 100
    })

iqr_summary = pd.DataFrame(iqr_summary)

iqr_summary

,Column,Q1,Q3,IQR,LowerBound,UpperBound,OutlierCount,OutlierPercent
0,ClosePrice,575000.0,1300000.0,725000.0,-512500.0,2387500.0,30715,7.421717
1,LivingArea,1248.0,2219.0,971.0,-208.5,3675.5,18207,4.399388
2,DaysOnMarket,8.0,48.0,40.0,-52.0,108.0,31772,7.677122


Check and flag for outliers that are in more than one of the listed columns.

In [24]:
# check whether each observation is an outlier in any numeric column
outlier_columns = [
    'outlier_closeprice',
    'outlier_livingarea',
    'outlier_daysonmarket'
]

sold['any_outlier_flag'] = sold[outlier_columns].any(axis = 1)

print('Total # of rows:', len(sold))
print('Rows with outliers:', sold['any_outlier_flag'].sum())

Total # of rows: 413853
Rows with outliers: 64971


## 4) Create new filtered dataset with removed flagged values

Double check whether any records still had invalid numeric values even though we cleaned it earlier (just in case something happened). Then, once confirmed that there are none, create the new sold filtered dataset.

In [27]:
sold['invalid_week7_flag'] = (
    (sold['ClosePrice'] <= 0) |
    (sold['LivingArea'] <= 0) |
    (sold['DaysOnMarket'] < 0)
)

print('Invalid numeric observations:', sold['invalid_week7_flag'].sum())

Invalid numeric observations: 0


In [28]:
sold_filtered = sold[~sold['any_outlier_flag'] & ~sold['invalid_week7_flag']].copy()

## 5) Compare # observations before and after filtering

Look at the dataset size and median values and also check if the filtering was completed correclty

In [32]:
# compare the number of rows before and after filtering
print('Rows before:', len(sold))
print('Rows after:', len(sold_filtered))
print('Rows removed:', len(sold) - len(sold_filtered))

# compare median values for the three columns
median_before = sold[numeric_columns].median()
median_after = sold_filtered[numeric_columns].median()

median_comparison = pd.DataFrame({
    'median_before': median_before,
    'median_after': median_after
})

median_comparison

Rows before: 413853
Rows after: 348882
Rows removed: 64971


,median_before,median_after
ClosePrice,823000.0,785000.0
LivingArea,1643.0,1569.0
DaysOnMarket,18.0,16.0


In [35]:
# check whether any outliers or invalid values remain
print('Remaining outliers:', sold_filtered['any_outlier_flag'].sum())
print('Remaining invalid values:', sold_filtered['invalid_week7_flag'].sum())

sold_filtered[numeric_columns].describe()

Remaining outliers: 0
Remaining invalid values: 0


,ClosePrice,LivingArea,DaysOnMarket
count,3.488820e+05,348780.000000,348882.000000
mean,8.988147e+05,1675.070403,26.270650
std,4.603330e+05,627.512075,25.518242
min,1.150000e+00,1.000000,0.000000
25%,5.650000e+05,1213.000000,7.000000
50%,7.850000e+05,1569.000000,16.000000
75%,1.150000e+06,2037.000000,39.000000
max,2.387500e+06,3675.000000,108.000000


### Save the new dataset and created reports

In [36]:
# missing values in the three numeric columns
missing_summary = pd.DataFrame({
    'MissingCount': sold[numeric_columns].isna().sum(),
    'MissingPercent': sold[numeric_columns].isna().mean() * 100
})

# save datasets and reports
output_dir = Path('../data/outlier_analysis')
report_dir = Path('../data/reports/week7_reports')

output_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

sold.to_csv(output_dir / 'sold_outliers_flagged.csv', index=False)
sold_filtered.to_csv(output_dir / 'sold_outliers_filtered.csv', index=False)

iqr_summary.to_csv(report_dir / 'iqr_summary.csv', index=False)
missing_summary.to_csv(report_dir / 'missing_summary.csv')

print('Week 7 files saved!')

Week 7 files saved!
